In [18]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


# Ollama API Client Example
This notebook shows how to call the Ollama service hosted at `https://ollama.lourie.info`.

**Prerequisites**: Install the `requests` library:
```bash
pip install requests
```

In [19]:
import requests
import json
import sys
import os
import time
from typing import Optional
from tqdm import tqdm

# Base URL of your Ollama API
BASE_URL = "https://ollama.lourie.info"

#Read credentials from file
with open('Ollama_Creds.json', 'r') as f:
    creds = json.load(f)
    f.close()



# If you set up HTTP Basic Authentication, provide credentials
auth = (creds['User'], creds['Password'])  # uncomment and fill if needed
#auth = None    #This has been disabled in nginx at ollama.lourie.info

### 1. List Available Models

In [47]:
response = requests.get(f"{BASE_URL}/api/tags", auth=auth)
if response.status_code == 200:
    models = response.json()
    print("Available models:")
    for model in models.get('models', []):
        print(f" - {model['name']}")
else:
    print(f"Error: {response.status_code} - {response.text}")

Available models:
 - qwen3.5:9b
 - deepseek-r1:8b
 - llama3.2:1b


### 2. Generate Text (Streaming Off)
Send a prompt and receive the full response.

In [13]:
payload = {
    "model": "llama3.2:1b",  # replace with a model you have
    "prompt": "When did the Great October Revolution happen?",
    "stream": False
}

response = requests.post(f"{BASE_URL}/api/generate", json=payload, auth=auth)
if response.status_code == 200:
    result = response.json()
    print("Response:", result['response'])
else:
    print(f"Error: {response.status_code} - {response.text}")

Response: The Great October Revolution, also known as the Bolshevik Revolution or October Revolution, occurred on October 25, 1917. It was a pivotal event in modern history that led to the overthrow of the Russian monarchy and the establishment of the world's first socialist state under the leadership of Vladimir Lenin and the Communist Party.


### 3. Stream Text Generation
Process the response as it arrives (useful for long outputs).

In [11]:
payload = {
    "model": "llama3.2:1b",
    "prompt": "Explain quantum computing in simple terms.",
    "stream": True
}

with requests.post(f"{BASE_URL}/api/generate", json=payload, auth=auth, stream=True) as r:
    if r.status_code == 200:
        for line in r.iter_lines():
            if line:
                chunk = json.loads(line)
                print(chunk.get('response', ''), end='')
                if chunk.get('done'):
                    break
        print()  # newline
    else:
        print(f"Error: {r.status_code} - {r.text}")

Quantum computing is a way of processing information that's different from regular computers. Here's a simplified explanation:

**Regular computers use "bits"**

Bits are like tiny light switches that can be either on (1) or off (0). They're the basic building blocks of digital data.

**Quantum computers use "qubits"**

Qubits (quantum bits) are different because they can be both on and off at the same time! This property is called superposition. It's like having a light switch that's always on, but you can also turn it on and off simultaneously.

Think of qubits like coins in a jar. Each coin represents a bit of information (0 or 1). In a regular computer, you flip each coin separately to get either 0 or 1. But with quantum computers, you can keep all the coins in the same jar, and they'll be both on and off at the same time!

**Quantum computers have many potential uses**

Quantum computers can help us solve problems that are too big for regular computers to handle. Some examples inc

### 4. Chat Completion
Use the chat endpoint for conversational models.

In [3]:
payload = {
    "model": "llama3.2:1b",
    "messages": [
        {"role": "user", "content": "Hello, how are you?"}
    ],
    "stream": False
}

response = requests.post(f"{BASE_URL}/api/chat", json=payload, auth=auth)
if response.status_code == 200:
    result = response.json()
    print("Assistant:", result['message']['content'])
else:
    print(f"Error: {response.status_code} - {response.text}")

Assistant: I'm doing well, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help you with any questions or topics you'd like to discuss. How about you? How's your day going so far?


4.A. Model pulling (Simple).

In [3]:
def pull_ollama_model(model_name: str, username: str = None, password: str = None, 
                      base_url: str = "http://ollama.lourie.info"):
    """
    Simple function to pull an ollama model with authentication
    """
    url = f"{base_url}/api/pull"
    
    payload = {
        "name": model_name,
        "stream": True
    }
    
    auth = (username, password) if username and password else None
    
    print(f"Pulling model: {model_name}")
    print("-" * 50)
    
    try:
        response = requests.post(url, json=payload, auth=auth, stream=True)
        response.raise_for_status()
        
        for line in response.iter_lines():
            if line:
                data = json.loads(line)
                if "status" in data:
                    print(f"  {data['status']}")
                if "error" in data:
                    print(f"  Error: {data['error']}")
                    return False
        
        print("-" * 50)
        print(f"✅ Model {model_name} pulled successfully")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error: {e}")
        return False

4.A1. Model pulling (with retries and progress bar).

In [41]:
#!/usr/bin/env python3
"""
Advanced Ollama model pull script with retries and progress bars.
Handles unstable connections gracefully.


import requests
import json
import sys
import os
import time
from typing import Optional
from tqdm import tqdm
"""

# Try to import tqdm, provide installation hint if missing
try:
    from tqdm import tqdm
except ImportError:
    print("This script requires 'tqdm' for progress bars.")
    print("Install it with: pip install tqdm")
    sys.exit(1)

# Default Ollama server URL – change this if needed
DEFAULT_OLLAMA_URL = "http://ollama.lourie.info"

class OllamaPuller:
    def __init__(self, base_url=BASE_URL, username=None, password=None, max_retries=5, retry_delay=5):
        # Allow override via environment variable OLLAMA_URL, otherwise use default
        self.base_url = base_url or os.environ.get('OLLAMA_URL', DEFAULT_OLLAMA_URL)
        self.auth = (username, password) if username and password else None
        self.session = requests.Session()
        if self.auth:
            self.session.auth = self.auth
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        print(f"🔧 Using Ollama server: {self.base_url}")

    def pull_model(self, model_name: str):
        """
        Pull a model with robust error handling and progress display
        """
        print(f"\n🚀 Starting pull for model: {model_name}")
        print("=" * 60)
        
        # First, get the model manifest to know what blobs we need
        manifest_url = f"{self.base_url}/api/pull"
        print(f"DEBUG: manifest_url = {manifest_url}")
        manifest_payload = {
            "name": model_name,
            "stream": False  # Get manifest first without streaming
        }
        
        try:
            # Get manifest to see what needs to be downloaded
            print("📋 Fetching model manifest...")
            response = self.session.post(manifest_url, json=manifest_payload, timeout=30)
            response.raise_for_status()
            
            manifest_data = response.json()
            
            # Check if model already exists
            if "status" in manifest_data and manifest_data["status"] == "success":
                print(f"✅ Model {model_name} already exists")
                return True
            
            # Parse the manifest to get layers
            if "layers" not in manifest_data:
                print("❌ Unexpected response format")
                return False
            
            layers = manifest_data["layers"]
            total_layers = len(layers)
            print(f"📦 Model consists of {total_layers} layers")
            
            # Download each layer with retries
            for idx, layer in enumerate(layers, 1):
                digest = layer["digest"]
                size = layer.get("size", 0)
                
                print(f"\n📥 Layer {idx}/{total_layers}: {digest[:12]}... ({self._format_size(size)})")
                
                # Download the blob with retries
                success = self._download_blob_with_retry(model_name, digest, size)
                if not success:
                    print(f"❌ Failed to download layer {digest}")
                    return False
            
            print("\n" + "=" * 60)
            print(f"✅ Model {model_name} pulled successfully!")
            return True
            
        except requests.exceptions.RequestException as e:
            print(f"❌ Network error: {e}")
            return False
    
    def _download_blob_with_retry(self, model_name: str, digest: str, total_size: int) -> bool:
        """
        Download a single blob with retry logic and resume support
        """
        blob_url = f"{self.base_url}/api/blobs/{digest}"
        headers = {}
        
        # Check if we already have part of this file (for resume)
        temp_file = f"/tmp/ollama_blob_{digest.replace(':', '_')}"
        mode = 'wb'
        start_byte = 0
        
        if os.path.exists(temp_file):
            start_byte = os.path.getsize(temp_file)
            if start_byte < total_size:
                headers['Range'] = f'bytes={start_byte}-'
                mode = 'ab'
                print(f"  ⏯️ Resuming from {self._format_size(start_byte)}")
            elif start_byte == total_size:
                print(f"  ✅ Layer already downloaded completely")
                return True
        
        for attempt in range(1, self.max_retries + 1):
            try:
                print(f"  🔄 Attempt {attempt}/{self.max_retries}...")
                
                response = self.session.get(
                    blob_url,
                    headers=headers,
                    stream=True,
                    timeout=(30, 60)  # (connect timeout, read timeout)
                )
                response.raise_for_status()
                
                # Check if server accepted range request
                if 'Range' in headers and response.status_code != 206:
                    print("  ⚠️ Server doesn't support resume, starting over")
                    mode = 'wb'
                    start_byte = 0
                
                # Download with progress bar
                with open(temp_file, mode) as f:
                    with tqdm(
                        total=total_size,
                        initial=start_byte,
                        unit='B',
                        unit_scale=True,
                        desc="  Progress",
                        bar_format='{l_bar}{bar:30}{r_bar}{bar:-10b}'
                    ) as pbar:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))
                
                # Verify size
                final_size = os.path.getsize(temp_file)
                if final_size == total_size:
                    # Move to final location or just indicate success
                    print(f"  ✅ Layer downloaded successfully")
                    # Clean up temp file if needed (ollama will manage its own storage)
                    os.unlink(temp_file)
                    return True
                else:
                    print(f"  ⚠️ Size mismatch: got {final_size}, expected {total_size}")
                    if attempt < self.max_retries:
                        wait_time = self.retry_delay * attempt
                        print(f"  ⏳ Waiting {wait_time}s before retry...")
                        time.sleep(wait_time)
                        # Update start_byte for next attempt
                        start_byte = final_size
                        headers['Range'] = f'bytes={start_byte}-'
                        continue
                    
            except requests.exceptions.RequestException as e:
                print(f"  ⚠️ Attempt {attempt} failed: {e}")
                if attempt < self.max_retries:
                    wait_time = self.retry_delay * attempt
                    print(f"  ⏳ Waiting {wait_time}s before retry...")
                    time.sleep(wait_time)
                    # If we have a partial file, try to resume
                    if os.path.exists(temp_file):
                        start_byte = os.path.getsize(temp_file)
                        if start_byte > 0 and start_byte < total_size:
                            headers['Range'] = f'bytes={start_byte}-'
                            print(f"  ⏯️ Will resume from {self._format_size(start_byte)}")
                else:
                    print(f"  ❌ Max retries exceeded")
        
        return False
    
    def _format_size(self, bytes_size: int) -> str:
        """Format bytes to human readable"""
        if not bytes_size:
            return "0 B"
        for unit in ['B', 'KB', 'MB', 'GB']:
            if bytes_size < 1024.0:
                return f"{bytes_size:.1f} {unit}"
            bytes_size /= 1024.0
        return f"{bytes_size:.1f} TB"

def main():
    import argparse
    
    parser = argparse.ArgumentParser(description="Robust Ollama model puller")
    parser.add_argument('--user', help='Username (or set OLLAMA_USER env var)')
    parser.add_argument('--password', help='Password (or set OLLAMA_PASSWORD env var)')
    parser.add_argument('--max-retries', type=int, default=5, 
                       help='Maximum retry attempts per layer (default: 5)')
    parser.add_argument('--retry-delay', type=int, default=5,
                       help='Base delay between retries in seconds (default: 5)')
    parser.add_argument('model', help='Model name to pull (e.g., qwen3.5:9b)')
    
    args = parser.parse_args()
    
    username = args.user or os.environ.get('OLLAMA_USER')
    password = args.password or os.environ.get('OLLAMA_PASSWORD')
    
    if not username or not password:
        print("Error: Authentication required. Provide credentials via --user/--password or OLLAMA_USER/OLLAMA_PASSWORD env vars.")
        sys.exit(1)
    
    puller = OllamaPuller(
        username=username,
        password=password,
        max_retries=args.max_retries,
        retry_delay=args.retry_delay
    )
    
    success = puller.pull_model(args.model)
    sys.exit(0 if success else 1)

"""
if __name__ == "__main__":
    main()
    """

'\nif __name__ == "__main__":\n    main()\n    '

In [39]:
!unset OLLAMA_URL
!export OLLAMA_URL="https://ollama.lourie.info"

In [44]:
#pull_ollama_model(model_name: "lfm2:24b", creds['User'], creds['Password'], base_url: str = "http://ollama.lourie.info")
#pull_ollama_model("qwen3.5:9b", creds['User'], creds['Password'], BASE_URL)
puller = OllamaPuller(BASE_URL,
    creds['User'],
    creds['Password'],
    max_retries=5,
    retry_delay=5
    )
print(puller.pull_model("qwen3.5:9b"))

🔧 Using Ollama server: https://ollama.lourie.info

🚀 Starting pull for model: qwen3.5:9b
DEBUG: manifest_url = https://ollama.lourie.info/api/pull
📋 Fetching model manifest...
✅ Model qwen3.5:9b already exists
True


4.B. Model management: memory usage, cleanup

In [6]:
'''
#!/usr/bin/env python3
"""
Check memory usage of ollama models via API
"""

import requests
import json
import sys
import os
'''

def check_model_memory(base_url="http://ollama.lourie.info", username=None, password=None):
    """
    Check currently loaded models and their memory usage
    """
    url = f"{base_url}/api/ps"
    auth = (username, password) if username and password else None
    
    try:
        response = requests.get(url, auth=auth, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        models = data.get("models", [])
        
        if not models:
            print("No models currently loaded in memory.")
            return
        
        print("\n" + "="*70)
        print("MODELS CURRENTLY IN MEMORY")
        print("="*70)
        
        for model in models:
            name = model.get("name", "Unknown")
            model_id = model.get("model", "Unknown")
            size = model.get("size", 0)
            processor = model.get("processor", "Unknown")
            until = model.get("until", "Unknown")
            
            # Format size to human-readable
            size_str = format_bytes(size)
            
            print(f"Model: {name}")
            print(f"  ID: {model_id}")
            print(f"  Memory: {size_str}")
            print(f"  Processor: {processor}")
            print(f"  Loaded until: {until}")
            print()
            
    except requests.exceptions.RequestException as e:
        print(f"Error checking memory: {e}")
        return False

def format_bytes(bytes_size):
    """Format bytes to human readable"""
    if not bytes_size:
        return "0 B"
    
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} PB"

def get_detailed_model_info(model_name, base_url="http://ollama.lourie.info", username=None, password=None):
    """
    Get detailed information about a specific model
    """
    url = f"{base_url}/api/show"
    auth = (username, password) if username and password else None
    
    payload = {
        "model": model_name
    }
    
    try:
        response = requests.post(url, json=payload, auth=auth, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        print(f"\nDetailed info for {model_name}:")
        print(json.dumps(data, indent=2))
        return data
        
    except requests.exceptions.RequestException as e:
        print(f"Error getting model details: {e}")
        return None
'''
if __name__ == "__main__":
    username = os.environ.get('OLLAMA_USER')
    password = os.environ.get('OLLAMA_PASSWORD')
    
    if not username or not password:
        print("Please set OLLAMA_USER and OLLAMA_PASSWORD environment variables")
        sys.exit(1)
    
    check_model_memory(username=username, password=password)
    
    # Optionally check a specific model
    # if len(sys.argv) > 1:
    #     get_detailed_model_info(sys.argv[1], username=username, password=password)
    '''

'\nif __name__ == "__main__":\n    username = os.environ.get(\'OLLAMA_USER\')\n    password = os.environ.get(\'OLLAMA_PASSWORD\')\n\n    if not username or not password:\n        print("Please set OLLAMA_USER and OLLAMA_PASSWORD environment variables")\n        sys.exit(1)\n\n    check_model_memory(username=username, password=password)\n\n    # Optionally check a specific model\n    # if len(sys.argv) > 1:\n    #     get_detailed_model_info(sys.argv[1], username=username, password=password)\n    '

In [46]:
check_model_memory(BASE_URL, creds['User'], creds['Password'])

No models currently loaded in memory.


4.C. Unloading models from memory

In [8]:
'''
#!/usr/bin/env python3
"""
Unload a model from ollama memory via API
"""

import requests
import json
import sys
import os
'''

def unload_model(model_name, base_url="http://ollama.lourie.info", username=None, password=None):
    """
    Unload a specific model from memory
    """
    url = f"{base_url}/api/generate"
    auth = (username, password) if username and password else None
    
    payload = {
        "model": model_name,
        "keep_alive": 0  # This tells ollama to unload immediately
    }
    
    try:
        print(f"Unloading model: {model_name}...")
        response = requests.post(url, json=payload, auth=auth, timeout=30)
        response.raise_for_status()
        
        print(f"✅ Model {model_name} unloaded successfully")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error unloading model: {e}")
        return False

def unload_all_models(base_url="http://ollama.lourie.info", username=None, password=None):
    """
    Attempt to unload all models by getting the list and unloading each
    """
    # First get list of loaded models
    ps_url = f"{base_url}/api/ps"
    auth = (username, password) if username and password else None
    
    try:
        response = requests.get(ps_url, auth=auth, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        models = data.get("models", [])
        
        if not models:
            print("No models currently loaded.")
            return True
        
        print(f"Found {len(models)} loaded models. Unloading...")
        
        for model in models:
            model_name = model.get("name")
            if model_name:
                unload_model(model_name, base_url, username, password)
        
        print("✅ All models unloaded")
        return True
        
    except requests.exceptions.RequestException as e:
        print(f"Error checking loaded models: {e}")
        return False

'''
if __name__ == "__main__":
    username = os.environ.get('OLLAMA_USER')
    password = os.environ.get('OLLAMA_PASSWORD')
    
    if not username or not password:
        print("Please set OLLAMA_USER and OLLAMA_PASSWORD environment variables")
        sys.exit(1)
    
    if len(sys.argv) < 2:
        print("Usage:")
        print("  python unload_model.py <model_name>  - Unload specific model")
        print("  python unload_model.py --all          - Unload all models")
        sys.exit(1)
    
    if sys.argv[1] == "--all":
        unload_all_models(username=username, password=password)
    else:
        unload_model(sys.argv[1], username=username, password=password)
        '''

'\nif __name__ == "__main__":\n    username = os.environ.get(\'OLLAMA_USER\')\n    password = os.environ.get(\'OLLAMA_PASSWORD\')\n\n    if not username or not password:\n        print("Please set OLLAMA_USER and OLLAMA_PASSWORD environment variables")\n        sys.exit(1)\n\n    if len(sys.argv) < 2:\n        print("Usage:")\n        print("  python unload_model.py <model_name>  - Unload specific model")\n        print("  python unload_model.py --all          - Unload all models")\n        sys.exit(1)\n\n    if sys.argv[1] == "--all":\n        unload_all_models(username=username, password=password)\n    else:\n        unload_model(sys.argv[1], username=username, password=password)\n        '

In [9]:
unload_model("deepseek-r1:8b", BASE_URL, creds['User'], creds['Password'])

Unloading model: deepseek-r1:8b...
✅ Model deepseek-r1:8b unloaded successfully


True

5. Selenium

In [3]:
from fake_useragent import UserAgent
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium_stealth import stealth
import time
import os

In [10]:
def get_random_chrome_user_agent():
    user_agent = UserAgent(browsers='mozilla', os='Linux', platforms='pc')
    return user_agent.random

In [9]:
def create_driver(user_id=1):
    options = Options()
    options.add_argument("start-maximized")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)

    script_dir = os.path.dirname(os.path.abspath(__file__))
    base_directory = os.path.join(script_dir, 'users')
    user_directory = os.path.join(base_directory, f'user_{user_id}')

    options.add_argument(f'user-data-dir={user_directory}')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    options.add_argument('--no-sandbox')
    # options.add_argument('--headless')

    driver = webdriver.Chrome(options=options)
    ua = get_random_chrome_user_agent()
    stealth(driver=driver,
            user_agent=ua,
            languages=["ru-RU", "ru"],
            vendor="Google Inc.",
            platform="Ubuntu64",
            webgl_vendor="Intel Inc.",
            renderer="Intel Iris OpenGL Engine",
            fix_hairline=True,
            run_on_insecure_origins=True
            )

    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        'source': '''
            delete window.cdc_adoQpoasnfa76pfcZLmcfl_Array;
            delete window.cdc_adoQpoasnfa76pfcZLmcfl_Promise;
            delete window.cdc_adoQpoasnfa76pfcZLmcfl_Symbol;
      '''
    })
    return driver

In [12]:
def main_login(user_id=1):
    driver = create_driver(user_id)
    driver.get('Ccылка на любой сайт')
    time.sleep(350)

5.1 Попробуем пока без fake user agent

In [4]:
import time
import re
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass, field, asdict
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import os

In [5]:
# -------------------- Data Schema --------------------
@dataclass
class Product:
    """Flexible product data container."""
    name: str
    price: float
    url: str
    # Optional fields
    currency: str = "RUB"
    rating: Optional[float] = None      # e.g., 4.5
    review_count: Optional[int] = None
    keyword_value: Optional[str] = None  # e.g., width "3 м"
    raw_data: Dict[str, Any] = field(default_factory=dict)  # any extra attributes

    def to_dict(self):
        return {k: v for k, v in asdict(self).items() if v is not None}

In [5]:
class Browser:
    def __init__(self, headless: bool = True, chrome_binary_path: str = None):
        self.headless = headless
        self.chrome_binary_path = chrome_binary_path
        self.driver = None

    def __enter__(self):
        options = Options()
        
        # Headless mode (modern Chrome uses --headless=new)
        if self.headless:
            options.add_argument("--headless=new")
        
        # Additional options to avoid detection and improve stability
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--disable-blink-features=AutomationControlled")
        
        # Set realistic user agent
        options.add_argument(
            "user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        
        # Exclude the "enable-automation" switch to hide automation
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        
        # If binary path is provided (e.g., for Chromium), set it
        if self.chrome_binary_path:
            options.binary_location = self.chrome_binary_path
        
        # If chromedriver is not in PATH, you can specify its path via Service
        # service = Service("/path/to/chromedriver")
        # self.driver = webdriver.Chrome(service=service, options=options)
        # Otherwise:
        self.driver = webdriver.Chrome(options=options)
        self.driver.implicitly_wait(10)
        return self.driver

    def __exit__(self, *args):
        if self.driver:
            self.driver.quit()

In [6]:
#Simple testing
with Browser(headless=True) as driver:
    driver.get("https://ollama.lourie.info")
    print("Page title:", driver.title)

Page title: 


In [12]:
def extract_product_blocks(soup: BeautifulSoup) -> list:
    """
    Find potential product blocks using heuristics:
    - Look for divs with classes containing 'product', 'item', 'card', etc.
    - Or find elements that contain a price pattern and are not too large.
    """
    # Try common class name patterns first
    potential_blocks = soup.find_all(['div', 'li', 'article'], 
                                     class_=re.compile(r'product|item|card|teplitsa|offer', re.I))
    
    if not potential_blocks:
        # Fallback: find all elements that contain a price
        price_pattern = re.compile(r'\d{1,3}(?:\s?\d{3})*\s*руб', re.I)
        all_elements = soup.find_all(['div', 'li', 'article'])
        potential_blocks = []
        for elem in all_elements:
            if price_pattern.search(elem.get_text()) and len(elem.get_text(strip=True)) < 1000:
                potential_blocks.append(elem)
    
    # Deduplicate blocks (some might be nested)
    unique_blocks = []
    for block in potential_blocks:
        # If this block is contained within another block we already have, skip it
        if not any(block in parent for parent in unique_blocks):
            unique_blocks.append(block)
    
    print(f"Found {len(unique_blocks)} potential product blocks")
    return unique_blocks

In [50]:
def extract_product_with_llm(block_html: str, ollama_url: str, auth: tuple, timeout: int = 180) -> Optional[Dict]:
    """
    Send a product block to Ollama and parse the JSON response.
    Now includes price cleaning and validation.
    """
    # Clean the HTML to reduce tokens
    block_soup = BeautifulSoup(block_html, 'html.parser')
    for script in block_soup(["script", "style"]):
        script.decompose()
    clean_text = block_soup.get_text(separator=' ', strip=True)
    
    prompt = f"""
    Extract product information from the following text and return ONLY a JSON object with these fields:
    - name: the product name
    - price: the current price as a number (without currency symbol, just digits and decimal point)
    - currency: always "RUB"
    - width: the width in meters if mentioned, otherwise null
    - rating: the average rating as a number (if available)
    - review_count: number of reviews as integer (if available)
    
    Text:
    {clean_text[:3000]}
    
    JSON:
    """
    
    payload = {
        "model": "deepseek-r1:8b",
        "prompt": prompt,
        "stream": False,
        "format": "json"
    }

    print(f"Text sent to LLM: {clean_text[:3000]}")
    
    try:
        response = requests.post(f"{ollama_url}/api/generate", json=payload, auth=auth, timeout=timeout)
        if response.status_code == 200:
            print(f"Response: {response}")
            result = response.json()
            json_str = result['response'].strip()
            # Remove markdown fences if present
            if json_str.startswith('```json'):
                json_str = json_str[7:]
            if json_str.endswith('```'):
                json_str = json_str[:-3]
            product_data = json.loads(json_str)
            
            # Clean price: remove any non-numeric characters except decimal point
            price_raw = product_data.get('price')
            if price_raw is not None:
                # Convert to string and clean
                price_str = str(price_raw)
                # Remove spaces, commas, and other non-numeric characters (keep digits and dot)
                cleaned = re.sub(r'[^\d.]', '', price_str)
                # Handle cases like "1.234,56" -> we assume dot as decimal separator after cleaning
                if cleaned:
                    try:
                        product_data['price'] = float(cleaned)
                    except ValueError:
                        product_data['price'] = None
                else:
                    product_data['price'] = None
            
            # Validate required fields
            if product_data.get('name') and product_data.get('price') is not None:
                return product_data
            else:
                print("Missing required fields (name or price)")
        else:
            print(f"Ollama error: {response.status_code}")
    except requests.Timeout:
        print(f"Timeout after {timeout}s for block")
    except Exception as e:
        print(f"Error processing block: {e}")
    
    return None

In [48]:
def extract_products_with_llm(soup: BeautifulSoup, url: str, ollama_url: str, auth: tuple, timeout: int = 180) -> list:
    """
    Extract products from a page using LLM. Skips blocks where price is missing.
    """
    blocks = extract_product_blocks(soup)
    products = []
    
    for idx, block in enumerate(blocks):
        print(f"Processing block {idx+1}/{len(blocks)}")
        block_html = str(block)
        data = extract_product_with_llm(block_html, ollama_url, auth, timeout=timeout)
        
        if data:
            # Now price is guaranteed to be a float (or None, but we already checked)
            product = {
                'name': data.get('name'),
                'price': data['price'],  # directly use the cleaned float
                'currency': data.get('currency', 'RUB'),
                'url': url,
                'keyword_value': data.get('width'),
                'rating': data.get('rating'),
                'review_count': data.get('review_count'),
                'raw_data': data
            }
            products.append(product)
    
    print(f"Extracted {len(products)} valid products")
    return products

In [39]:
def fetch_page(url: str, headless: bool = True, wait_time: int = 5) -> BeautifulSoup:
    """
    Load page with Selenium, wait `wait_time` seconds, return BeautifulSoup.
    """
    with Browser(headless=headless) as driver:
        print(f"🌐 Loading {url} ...")
        driver.get(url)
        # Give the page a moment to render dynamic content
        time.sleep(wait_time)
        html = driver.page_source
    return BeautifulSoup(html, 'html.parser')

In [34]:
OLLAMA_URL = "https://ollama.lourie.info"

URL = "https://teplitsa-rus.ru/"

In [35]:
# Fetch page
soup = fetch_page('https://teplitsa-rus.ru/item/2-teplica-arochnaya-25m/', headless=True)
print(f"Page title: {soup.title.text}")

🌐 Loading https://teplitsa-rus.ru/item/2-teplica-arochnaya-25m/ ...
Page title: Теплица Боярская 2.5м - купить в Москве на заводе 


In [51]:
# Extract products with LLM
products = extract_products_with_llm(soup, URL, OLLAMA_URL, auth)

Found 10 potential product blocks
Processing block 1/10
Text sent to LLM: Каталог О Нас Гарантия на теплицы Доставка и Сборка Частые вопросы Отзывы Наши работы Контакты
Response: <Response [200]>
Missing required fields (name or price)
Processing block 2/10
Text sent to LLM: Арочные теплицы Прямостенные теплицы Каплевидные теплицы Двускатные теплицы Теплицы по Миттлайдеру Пристенные теплицы Раздвижные теплицы Промышленные теплицы Навесы Грядки оцинкованные
Response: <Response [200]>
Missing required fields (name or price)
Processing block 3/10
Text sent to LLM: Главная > Арочные теплицы > Теплица Боярская 2.5м Теплица Боярская 2.5м Поликарбонат с защитой от ультрафиолета Цельные дуги из стальной замкнутой трубы Каркас из оцинкованной внутри и снаружи стали Защита от коррозии и ржавчины Краб-система на 4 болта Разработано для сурового Российского климата Гарантия до 15 лет Каркас : оцинкованная труба 20х20 мм Ширина : 2.5 м Высота : 2.1 м Снеговая нагрузка : 227 кг/м2 Горизонтальные стя

In [41]:
len(products)

4

In [53]:
for product in products:
    print(product)

{'name': 'Теплица Боярская 2.5м', 'price': 17990.0, 'currency': 'RUB', 'url': 'https://teplitsa-rus.ru/', 'keyword_value': 2.5, 'rating': None, 'review_count': None, 'raw_data': {'name': 'Теплица Боярская 2.5м', 'price': 17990.0, 'currency': 'RUB', 'width': 2.5, 'rating': None, 'review_count': None}}
{'name': 'Product Name', 'price': 999.99, 'currency': 'RUB', 'url': 'https://teplitsa-rus.ru/', 'keyword_value': None, 'rating': 4.5, 'review_count': 123, 'raw_data': {'name': 'Product Name', 'price': 999.99, 'currency': 'RUB', 'width': None, 'rating': 4.5, 'review_count': 123}}
{'name': 'Поликарбонат с защитой от ультрафиолета', 'price': 17990.0, 'currency': 'RUB', 'url': 'https://teplitsa-rus.ru/', 'keyword_value': 2.5, 'rating': None, 'review_count': None, 'raw_data': {'name': 'Поликарбонат с защитой от ультрафиолета', 'price': 17990.0, 'currency': 'RUB', 'width': 2.5, 'rating': None, 'review_count': None}}
{'name': 'Капельный полив механический X', 'price': 1890.0, 'currency': 'RUB', '

In [54]:
payload = {
    "model": "deepseek-r1:8b",
    "messages": [
        {"role": "user", "content": "Как дела? Ты там живой?"}
    ],
    "stream": False
}

response = requests.post(f"{BASE_URL}/api/chat", json=payload, auth=auth)
if response.status_code == 200:
    result = response.json()
    print("Assistant:", result['message']['content'])
else:
    print(f"Error: {response.status_code} - {response.text}")

Assistant: Привет! Да, всё в порядке, я "вроде да" жив и готов тебя выслушать или помочь с вопросом. 😊  
Хочешь сказать "привет" — значит, я работаю. Если что-то случилось или тебе нужна помощь — пиши, я всегда рядом.  
А ты как там, в своем мире? Всё в порядке?
